# Transformación del MC1 Graph: De crudo a centrado-en-artworks

Este notebook reproduce la transformación que aplica **Melody Way** (VAST 2025 Mini-Challenge 1) sobre el grafo crudo `MC1_graph.json` para obtener `MC1_graph_artworks.json`.

**Idea general:** El JSON crudo viene como un grafo dirigido con `nodes` y `links` (edges). Para alimentar la visualización radial, en lugar de iterar miles de aristas en cada render, se *aplanan* los edges como **propiedades de los nodos**. Por ejemplo, un edge `(persona) --PerformerOf--> (canción)` se convierte en dos atributos:

- en la canción: se agrega la persona a `performedBy`
- en la persona: se agrega la canción a `contributedTo` y se registra el rol `Performer`

Además se computan:

- `lastPublishedYear` por artista (año más reciente entre sus artworks)
- `roles` consolidados por artista y por sello
- `influence` por año y `totalInfluence` para artistas, usando los pesos del paper:
    - `CoverOf` 1.0, `DirectlySamples` 0.9, `LyricalReferenceTo` 0.8, `InterpolatesFrom` 0.7, `InStyleOf` 0.6

El resultado es un único JSON sin `links`, donde cada nodo lleva toda la información de sus relaciones embebida.

## 1. Carga del grafo crudo

Cargamos el JSON original. Tiene cuatro secciones: metadata (`directed`, `multigraph`, `graph`), `nodes` y `links`. Los nodos vienen con cinco tipos: `Song`, `Album`, `Person`, `MusicalGroup`, `RecordLabel`. Los edges traen un `Edge Type` que indica la relación.

In [ ]:
import json
from collections import defaultdict, Counter

# Ajustá las rutas a donde tengas los archivos
INPUT_PATH = 'MC1_graph.json'
OUTPUT_PATH = 'MC1_graph_artworks_generated.json'

with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

print(f"Nodos: {len(raw['nodes'])}")
print(f"Edges: {len(raw['links'])}")
print(f"Tipos de nodo: {Counter(n['Node Type'] for n in raw['nodes'])}")
print(f"Tipos de edge: {Counter(e['Edge Type'] for e in raw['links'])}")

## 2. Constantes de la transformación

Definimos los conjuntos de tipos y las tablas de mapeo entre **tipos de edge** y los **campos** que generan en cada nodo:

- **INFLUENCE_TYPES**: edges de influencia entre artworks (o, en algunos casos especiales, entre artistas).
- **INFLUENCE_WEIGHTS**: pesos para calcular el score de influencia (heurística del paper).
- **ROLE_EDGE_MAP**: edges que asignan un rol creativo (`PerformerOf`, `ComposerOf`, `ProducerOf`, `LyricistOf`). Mapean a un campo en el artwork (`performedBy`, etc.) y a un rol en el artista (`Performer`, etc.).
- **LABEL_EDGE_MAP**: edges con sellos (`RecordedBy`, `DistributedBy`). Mapean a campos en el artwork y al sello, además del rol del sello (`Recorder`/`Distributor`).

In [ ]:
INFLUENCE_TYPES = {'CoverOf', 'DirectlySamples', 'LyricalReferenceTo', 'InterpolatesFrom', 'InStyleOf'}

INFLUENCE_WEIGHTS = {
    'CoverOf': 1.0,
    'DirectlySamples': 0.9,
    'LyricalReferenceTo': 0.8,
    'InterpolatesFrom': 0.7,
    'InStyleOf': 0.6,
}

# edge_type -> (campo en artwork, rol en artista)
ROLE_EDGE_MAP = {
    'PerformerOf': ('performedBy', 'Performer'),
    'ComposerOf':  ('composedBy',  'Composer'),
    'ProducerOf':  ('producedBy',  'Producer'),
    'LyricistOf':  ('lyricsBy',    'Lyricist'),
}

# edge_type -> (campo en artwork, campo en label, rol del label)
LABEL_EDGE_MAP = {
    'RecordedBy':    ('recordedBy',    'recordedArtwork',    'Recorder'),
    'DistributedBy': ('distributedBy', 'distributedArtwork', 'Distributor'),
}

ARTWORK_TYPES = {'Song', 'Album'}
ARTIST_TYPES  = {'Person', 'MusicalGroup'}

# Lookup por id para acceder rápido a propiedades del nodo
node_by_id = {n['id']: n for n in raw['nodes']}

## 3. Estructuras acumuladoras

Inicializamos los "campos extra" que vamos a llenar para cada nodo según su tipo. Los conservamos aparte del nodo crudo para no contaminar `raw` y para poder controlar el orden y la presencia de claves al final.

Para artistas también guardamos `artist_inf_in`: edges de influencia *artista→artista* (caso especial mencionado en el paper, ej. Sailor Shift). Esos generan una entrada de influencia con año `"Unknown"`.

In [ ]:
# Para cada artwork, listas de IDs por relación
artwork_extras = {
    n['id']: {
        'influencedBy': [],
        'producedBy':   [],
        'composedBy':   [],
        'lyricsBy':     [],
        'performedBy':  [],
        'recordedBy':   [],
        'distributedBy':[],
        'influenced':   [],
    }
    for n in raw['nodes'] if n.get('Node Type') in ARTWORK_TYPES
}

# Para cada artista (Person / MusicalGroup)
artist_extras = {
    n['id']: {
        'contributedTo':   [],   # IDs de artworks (o artistas, en casos extraños)
        'roles':           [],   # roles creativos consolidados
        'last_year':       None, # se calcula al final
        'artist_inf_in':   [],   # edges de influencia entrantes a nivel artista
    }
    for n in raw['nodes'] if n.get('Node Type') in ARTIST_TYPES
}

# Para cada sello
label_extras = {
    n['id']: {
        'recordedArtwork':    [],
        'distributedArtwork': [],
        'roles':              [],
    }
    for n in raw['nodes'] if n.get('Node Type') == 'RecordLabel'
}

def add_unique(lst, val):
    """Append val a lst sólo si todavía no está. Mantiene orden de inserción."""
    if val not in lst:
        lst.append(val)

## 4. Recorrido principal de edges

Iteramos cada edge una sola vez y, según su tipo, lo "aplanamos" en los campos correspondientes:

1. **Influencia entre artworks**: agrega al `influencedBy` del origen y al `influenced` del destino.
2. **Influencia hacia un artista**: la guardamos como pendiente; se traduce a un score "Unknown".
3. **Roles creativos** (`PerformerOf`, etc.):
    - Si la fuente es un artista, registra contribución y rol al artista, y agrega al artwork si el destino es artwork.
    - Si la fuente es un sello (caso raro: 26 sellos actúan como `Producer`), agrega el rol al sello y el sello al artwork.
4. **Edges con sello** (`RecordedBy`, `DistributedBy`): la mayoría apunta `artwork → label`, pero hay 103 invertidos (`label → artwork`). Manejamos ambos sentidos. Cuando el sello es la *fuente* de `RecordedBy/DistributedBy` no le añadimos rol `Recorder/Distributor` (sólo se cuenta en sentido canónico, replicando el comportamiento del archivo original).
5. **`MemberOf`**: relación persona→grupo, no genera campos derivados; se ignora en este paso.

In [ ]:
for e in raw['links']:
    et = e['Edge Type']
    s, t = e['source'], e['target']
    s_t = node_by_id[s].get('Node Type')
    t_t = node_by_id[t].get('Node Type')

    # 1 y 2: edges de influencia
    if et in INFLUENCE_TYPES:
        if s_t in ARTWORK_TYPES and t_t in ARTWORK_TYPES:
            # caso normal: artwork -> artwork
            add_unique(artwork_extras[s]['influencedBy'], t)
            add_unique(artwork_extras[t]['influenced'],   s)
        elif t_t in ARTIST_TYPES:
            # caso especial: alguien influye a un artista (Sailor, etc.)
            artist_extras[t]['artist_inf_in'].append(e)

    # 3: roles creativos
    elif et in ROLE_EDGE_MAP:
        artwork_field, role = ROLE_EDGE_MAP[et]
        if s_t in ARTIST_TYPES:
            # un artista contribuye con un rol
            add_unique(artist_extras[s]['contributedTo'], t)
            add_unique(artist_extras[s]['roles'], role)
            if t_t in ARTWORK_TYPES:
                add_unique(artwork_extras[t][artwork_field], s)
        elif s_t == 'RecordLabel':
            # caso raro: sellos que figuran como Producer (26 casos)
            add_unique(label_extras[s]['roles'], role)
            if t_t in ARTWORK_TYPES:
                add_unique(artwork_extras[t][artwork_field], s)

    # 4: edges con sello
    elif et in LABEL_EDGE_MAP:
        artwork_field, label_field, label_role = LABEL_EDGE_MAP[et]
        if s_t in ARTWORK_TYPES and t_t == 'RecordLabel':
            # dirección canónica: artwork -> label
            add_unique(artwork_extras[s][artwork_field], t)
            add_unique(label_extras[t][label_field], s)
            add_unique(label_extras[t]['roles'], label_role)
        elif s_t == 'RecordLabel' and t_t in ARTWORK_TYPES:
            # dirección invertida: 103 edges con label -> artwork
            # se enlaza el artwork pero NO se agrega Recorder/Distributor al sello
            add_unique(artwork_extras[t][artwork_field], s)
            add_unique(label_extras[s][label_field], t)
    # 5: MemberOf y otros: ignorar acá

print("Recorrido completado.")

## 5. `lastPublishedYear` por artista

Para cada artista, su "último año publicado" es el máximo `release_date` de sus `contributedTo` que sea numérico. Si el artista no contribuyó a ningún artwork con fecha válida, el campo se omite.

In [ ]:
for aid, d in artist_extras.items():
    years = []
    for art_id in d['contributedTo']:
        if art_id in node_by_id:
            y = node_by_id[art_id].get('release_date')
            try:
                years.append(int(y))
            except (ValueError, TypeError):
                pass
    d['last_year'] = max(years) if years else None

print("lastPublishedYear calculado para todos los artistas.")

## 6. Score de influencia por año

**Regla descubierta** (consistente con el paper y el código del visualizador):

Para cada artista A:
- Para cada artwork X al que A contribuye:
  - Sumar los pesos de las edges de influencia *entrantes* a X (otra obra cita/cubre/imita a X)…
  - **excluyendo aquellas cuya obra fuente también pertenece a A** (no contamos auto-citas).
  - Multiplicar ese subtotal por **2** (factor de escala usado en el archivo; en la UI se vuelve a dividir por 2 para mostrarlo).
  - Atribuirlo al `release_date` de X.
- Si A es destino de edges de influencia *artista→artista*, sumar esos pesos × 2 al año `"Unknown"`.

El `totalInfluence` es la suma de todos los scores anuales. Si el resultado es vacío, se omiten los campos `influence` y `totalInfluence`.

In [ ]:
# Mapeo artwork_id -> año de release (string) para mirar rápido
artwork_to_year = {
    n['id']: n.get('release_date', 'Unknown')
    for n in raw['nodes'] if n.get('Node Type') in ARTWORK_TYPES
}

# Pre-construir: para cada artwork, lista de edges de influencia entrantes
inc_inf_by_artwork = defaultdict(list)
for e in raw['links']:
    if e['Edge Type'] in INFLUENCE_TYPES:
        inc_inf_by_artwork[e['target']].append(e)

artist_influence = {}  # aid -> (lista de {year, score}, total)

for aid, d in artist_extras.items():
    score_by_year = defaultdict(float)
    my_arts = set(d['contributedTo'])

    # Influencia a través de los artworks del artista
    for art in my_arts:
        if art not in artwork_to_year:
            continue  # contribuyó a otro artista (caso muy raro), no es artwork
        year = artwork_to_year[art]
        # Suma de pesos entrantes, excluyendo auto-influencia
        inc = sum(
            INFLUENCE_WEIGHTS[edge['Edge Type']]
            for edge in inc_inf_by_artwork[art]
            if edge['source'] not in my_arts
        )
        if inc > 0:
            score_by_year[year] += inc * 2  # factor x2 (la UI divide por 2 al mostrar)

    # Influencia artista->artista (caso Sailor): se agrupa como año "Unknown"
    if d['artist_inf_in']:
        unk = sum(INFLUENCE_WEIGHTS[edge['Edge Type']] for edge in d['artist_inf_in']) * 2
        if unk > 0:
            score_by_year['Unknown'] += unk

    if score_by_year:
        # Ordenamos años numéricos asc, dejando "Unknown" al final
        def year_key(y):
            return (y == 'Unknown', y)
        sorted_years = sorted(score_by_year.keys(), key=year_key)
        cleaned = [{'year': y, 'score': round(score_by_year[y], 4)} for y in sorted_years]
        total = round(sum(item['score'] for item in cleaned), 4)
        artist_influence[aid] = (cleaned, total)

print(f"Artistas con influencia: {len(artist_influence)}")

## 7. Construcción del nodo final

Reglas observadas en `MC1_graph_artworks.json` para qué campos van vacíos vs omitidos:

**Songs / Albums**:
- `performedBy`, `recordedBy`, `distributedBy`, `influenced` → siempre presentes (aunque sean `[]`).
- `influencedBy`, `producedBy`, `composedBy`, `lyricsBy` → se omiten si quedarían vacíos.
- Las claves originales (`Node Type`, `name`, `release_date`, `genre`, `single`, `notable`, `id`, `notoriety_date`, `written_date`) se conservan en el orden en que aparecen en el JSON crudo.

**Persons / MusicalGroups**:
- `contributedTo`, `roles` → siempre presentes (aunque sean `[]`).
- `lastPublishedYear` → se omite si no hay años válidos.
- `influence`, `totalInfluence` → sólo si el artista tiene score > 0.
- `stage_name` → se conserva si venía en el crudo.

**RecordLabels**:
- `recordedArtwork`, `distributedArtwork`, `roles` → siempre presentes.

Construimos un nuevo nodo a partir del crudo y agregamos los campos derivados en el orden indicado.

In [ ]:
def build_artwork_node(raw_node):
    """Construye un nodo Song/Album con los campos derivados."""
    nid = raw_node['id']
    d = artwork_extras[nid]
    new_n = dict(raw_node)  # preserva orden y atributos originales

    # influencedBy se omite si está vacío
    if d['influencedBy']:
        new_n['influencedBy'] = list(d['influencedBy'])
    # roles creativos: se omiten si vacíos
    for fld in ['producedBy', 'composedBy', 'lyricsBy']:
        if d[fld]:
            new_n[fld] = list(d[fld])
    # performedBy: siempre presente
    new_n['performedBy'] = list(d['performedBy'])
    # recordedBy y distributedBy: siempre presentes
    new_n['recordedBy']   = list(d['recordedBy'])
    new_n['distributedBy']= list(d['distributedBy'])
    # influenced: siempre presente
    new_n['influenced']   = list(d['influenced'])
    return new_n


def build_artist_node(raw_node):
    """Construye un nodo Person/MusicalGroup con campos derivados."""
    nid = raw_node['id']
    d = artist_extras[nid]
    new_n = dict(raw_node)

    new_n['contributedTo'] = list(d['contributedTo'])
    if d['last_year'] is not None:
        new_n['lastPublishedYear'] = d['last_year']
    new_n['roles'] = list(d['roles'])
    if nid in artist_influence:
        inf_list, total = artist_influence[nid]
        new_n['influence'] = inf_list
        new_n['totalInfluence'] = total
    return new_n


def build_label_node(raw_node):
    """Construye un nodo RecordLabel con campos derivados."""
    nid = raw_node['id']
    d = label_extras[nid]
    new_n = dict(raw_node)
    new_n['recordedArtwork']    = list(d['recordedArtwork'])
    new_n['distributedArtwork'] = list(d['distributedArtwork'])
    new_n['roles']              = list(d['roles'])
    return new_n


new_nodes = []
for n in raw['nodes']:
    nt = n.get('Node Type')
    if nt in ARTWORK_TYPES:
        new_nodes.append(build_artwork_node(n))
    elif nt in ARTIST_TYPES:
        new_nodes.append(build_artist_node(n))
    elif nt == 'RecordLabel':
        new_nodes.append(build_label_node(n))
    else:
        new_nodes.append(dict(n))

print(f"Nodos generados: {len(new_nodes)}")

## 8. Armado del JSON final

El JSON destino conserva la metadata del crudo (`directed`, `multigraph`, `graph`) y la lista de nodos transformada. **No incluye `links`** porque toda la información relacional ya quedó embebida en cada nodo.

In [ ]:
output = {
    'directed':   raw.get('directed', True),
    'multigraph': raw.get('multigraph', True),
    'graph':      raw.get('graph', {'node_default': {}, 'edge_default': {}}),
    'nodes':      new_nodes,
}

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=None, separators=(', ', ': '))

print(f"Archivo escrito en: {OUTPUT_PATH}")

## 9. Validación contra el archivo original

Comparamos el resultado con `MC1_graph_artworks.json` para confirmar que el contenido es equivalente. Comparamos cada campo derivado tratando los arrays como conjuntos (ignorando el orden), y los scores de influencia con tolerancia numérica.

> **Nota:** El orden interno de los elementos dentro de los arrays (ej. el orden de IDs en `performedBy`) puede diferir del archivo original porque el script original parece haber usado una iteración específica que no es trivialmente reproducible. El **contenido** de cada array es idéntico.

In [ ]:
REFERENCE_PATH = 'MC1_graph_artworks.json'  # ajustá si está en otra ruta

try:
    with open(REFERENCE_PATH, 'r', encoding='utf-8') as f:
        reference = json.load(f)
except FileNotFoundError:
    print(f"No se encontró {REFERENCE_PATH}; se omite la validación.")
    reference = None

if reference is not None:
    assert len(new_nodes) == len(reference['nodes']), 'distinta cantidad de nodos'
    mismatches = 0
    for mine, theirs in zip(new_nodes, reference['nodes']):
        if mine['id'] != theirs['id']:
            print(f"Orden de id distinto: {mine['id']} vs {theirs['id']}")
            mismatches += 1
            continue
        for key in set(mine) | set(theirs):
            if key == 'influence':
                a = {(d['year'], round(d['score'], 4)) for d in mine.get(key, [])}
                b = {(d['year'], round(d['score'], 4)) for d in theirs.get(key, [])}
                if a != b:
                    mismatches += 1
            elif key == 'totalInfluence':
                if round(mine.get(key, 0), 4) != round(theirs.get(key, 0), 4):
                    mismatches += 1
            elif isinstance(mine.get(key), list) and isinstance(theirs.get(key), list):
                if sorted(mine.get(key, [])) != sorted(theirs.get(key, [])):
                    mismatches += 1
            else:
                # Para campos opcionales ausentes en uno u otro lado, compará valores 'normalizados'
                m_val = mine.get(key)
                t_val = theirs.get(key)
                if m_val != t_val:
                    # Tolerar un lado sin la clave y el otro con valor 'falsy' (lista vacía)
                    if not ((m_val in (None, [], '') and t_val in (None, [], ''))):
                        mismatches += 1
    total_fields = sum(len(set(m) | set(t)) for m, t in zip(new_nodes, reference['nodes']))
    print(f"Diferencias de contenido (ignorando orden de arrays): {mismatches} / ~{total_fields} campos")

## 10. Resumen de la transformación aplicada

1. **Aplanado de relaciones**: cada edge del grafo se proyecta como atributos de nodo. Resultado: el grafo deja de tener `links` y queda "todo en los nodos".
2. **Agregados creativos**: `contributedTo`, `roles` y `lastPublishedYear` resumen la trayectoria de cada artista.
3. **Agregados de sello**: `recordedArtwork`, `distributedArtwork`, `roles` resumen la actividad de cada sello.
4. **Influencia ponderada**: para cada artista se calcula un score por año (excluyendo auto-citas) y un `totalInfluence`. Se usa el factor ×2 que después la UI divide al mostrar el valor.
5. **Casos especiales** del dataset: edges `RecordedBy`/`DistributedBy` invertidos, sellos que actúan como `Producer`, e influencias artista→artista (Sailor Shift) → todos manejados con sus reglas particulares.
6. **Política de campos**: campos siempre presentes vs. omitidos cuando vacíos, según lo observado en el archivo de referencia.

Este JSON resultante es el que `Overview.jsx` consume para dibujar la galaxia de artistas y la línea de tiempo de artworks.